In [ ]:
# Step 1: Install & Import Libraries
!pip install -q earthengine-api geemap tensorflow torch

import ee
import geemap
import os
import datetime
import glob

print('Libraries imported successfully.')

In [ ]:
# Step 2: Authenticate and Initialize GEE
ee.Authenticate()

GEE_PROJECT_ID = 'your-gee-project-id' # Replace with your Google Cloud Project ID
ee.Initialize(project=GEE_PROJECT_ID)
print('Google Earth Engine Initialized!')

In [ ]:
# Step 3: AOI Definition & 5km Grid Subdivision 
FAISALABAD_BBOX = [72.70, 31.10, 73.50, 31.70] # [xmin, ymin, xmax, ymax]
aoi = ee.Geometry.BBox(*FAISALABAD_BBOX)

START_DATE = '2023-01-01'
END_DATE = '2024-12-31'
PATCH_SIZE = 256
SCALE = 10
MIN_CROPLAND_PERCENT = 70.0 # Min 70% cropland ratio per patch
DRIVE_FOLDER = 'GEE_Faisalabad_Crop_Dataset_Corrected'

print(f'Pipeline configured for Faisalabad ({START_DATE} to {END_DATE})')

In [ ]:
# Step 4: ESA WorldCover 10m Cropland Verification Masking (Correction 1)
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
cropland_mask = worldcover.select('map').eq(40).rename('cropland_mask') # Class 40 = Cropland

# Calculate local cropland density
cropland_density = cropland_mask.reduceNeighborhood(
    reducer=ee.Reducer.mean(),
    kernel=ee.Kernel.square(radius=128, units='pixels')
).rename('cropland_density')

print('Cropland density and verification mask generated.')

In [ ]:
# Step 5: MODIS QA Masking (Joining MOD09GQ 250m with MOD09GA 1km State QA) 
def mask_modis_qa(image):
    qa = image.select('state_1km')
    cloud_state = qa.bitwiseAnd(3).eq(0)       # bits 0-1: 00 = clear
    cloud_shadow = qa.bitwiseAnd(4).eq(0)      # bit 2: 0 = no shadow
    cirrus = qa.bitwiseAnd(24).eq(0)           # bits 3-4: 00 = no cirrus
    clear_mask = cloud_state.And(cloud_shadow).And(cirrus)
    return image.updateMask(clear_mask)

def process_modis(date_start, date_end):
    gq = ee.ImageCollection('MODIS/061/MOD09GQ').filterBounds(aoi).filterDate(date_start, date_end)
    ga = ee.ImageCollection('MODIS/061/MOD09GA').filterBounds(aoi).filterDate(date_start, date_end)
    
    filter_date = ee.Filter.equals(leftField='system:index', rightField='system:index')
    joined = ee.Join.inner().apply(gq, ga, filter_date)
    
    def combine_bands(pair):
        img_gq = ee.Image(pair.get('primary'))
        img_ga = ee.Image(pair.get('secondary'))
        masked_ga = mask_modis_qa(img_ga)
        
        red = img_gq.select('sur_refl_b01').updateMask(masked_ga.select('state_1km').notNull())
        nir = img_gq.select('sur_refl_b02').updateMask(masked_ga.select('state_1km').notNull())
        ndvi = nir.subtract(red).divide(nir.add(red)).rename('MODIS_NDVI')
        
        return red.rename('MODIS_RED').addBands(nir.rename('MODIS_NIR')).addBands(ndvi).set('system:time_start', img_gq.get('system:time_start'))
        
    return ee.ImageCollection(joined.map(combine_bands))

modis_col = process_modis(START_DATE, END_DATE)
print('MODIS QA-masked collection ready.')

In [ ]:
# Step 6: Sentinel-2 Multiband & S2Cloudless Integration
def process_s2(date_start, date_end):
    s2_sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi).filterDate(date_start, date_end)
    s2_prob = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filterBounds(aoi).filterDate(date_start, date_end)
    
    filter_date = ee.Filter.equals(leftField='system:index', rightField='system:index')
    joined = ee.Join.inner().apply(s2_sr, s2_prob, filter_date)
    
    def add_s2_features(pair):
        img_sr = ee.Image(pair.get('primary'))
        img_prob = ee.Image(pair.get('secondary'))
        
        scl = img_sr.select('SCL')
        clear_mask = scl.eq(4).or(scl.eq(5)) # Vegetation / Bare soil
        masked_sr = img_sr.updateMask(clear_mask)
        
        b2 = masked_sr.select('B2').rename('S2_B2')   # Blue
        b3 = masked_sr.select('B3').rename('S2_B3')   # Green
        b4 = masked_sr.select('B4').rename('S2_B4')   # Red
        b8 = masked_sr.select('B8').rename('S2_B8')   # NIR
        b11 = masked_sr.select('B11').rename('S2_B11') # SWIR1
        b12 = masked_sr.select('B12').rename('S2_B12') # SWIR2
        ndvi = masked_sr.normalizedDifference(['B8', 'B4']).rename('S2_NDVI')
        cloud_prob = img_prob.select('probability').rename('S2_cloud_prob')
        
        return ee.Image.cat([b2, b3, b4, b8, b11, b12, ndvi, cloud_prob]).set('system:time_start', img_sr.get('system:time_start'))
        
    return ee.ImageCollection(joined.map(add_s2_features))

s2_col = process_s2(START_DATE, END_DATE)
print('Sentinel-2 multiband collection ready.')

In [ ]:
# Step 7: Multiband Triplet Stacker
def build_triplet(s2_target_img):
    target_date = s2_target_img.date()
    s2_proj = s2_target_img.select('S2_NDVI').projection()
    
    # 1. MODIS Target Image (+/- 1 day window flexibility, Bilinear Resampling) (Corrections 3 & 10)
    modis_target_col = modis_col.filterDate(target_date.advance(-1, 'day'), target_date.advance(2, 'day'))
    if modis_target_col.size().getInfo() == 0:
        return None # Robust check (Correction 4)
        
    modis_target = modis_target_col.first().resample('bilinear').reproject(crs=s2_proj, scale=SCALE)
    
    # 2. Reference Sentinel-2 Image (-15 to -3 days)
    s2_ref_col = s2_col.filterDate(target_date.advance(-15, 'day'), target_date.advance(-3, 'day'))
    if s2_ref_col.size().getInfo() == 0:
        return None
        
    s2_ref = s2_ref_col.first()
    ref_date = s2_ref.date()
    
    # 3. MODIS Reference Image on reference date
    modis_ref_col = modis_col.filterDate(ref_date.advance(-1, 'day'), ref_date.advance(2, 'day'))
    if modis_ref_col.size().getInfo() == 0:
        return None
        
    modis_ref = modis_ref_col.first().resample('bilinear').reproject(crs=s2_proj, scale=SCALE)
    
    # 4. Calculate time_gap_days metadata (Correction 5)
    time_gap = target_date.difference(ref_date, 'day')
    time_gap_img = ee.Image.constant(time_gap).rename('time_gap_days')
    
    # Build Multiband Stack
    stack = ee.Image.cat([
        modis_target.select(['MODIS_RED', 'MODIS_NIR', 'MODIS_NDVI']).select(['MODIS_RED', 'MODIS_NIR', 'MODIS_NDVI'], ['M_target_R', 'M_target_NIR', 'M_target_NDVI']),
        modis_ref.select(['MODIS_RED', 'MODIS_NIR', 'MODIS_NDVI']).select(['MODIS_RED', 'MODIS_NIR', 'MODIS_NDVI'], ['M_ref_R', 'M_ref_NIR', 'M_ref_NDVI']),
        s2_ref.select(['S2_B2', 'S2_B3', 'S2_B4', 'S2_B8', 'S2_B11', 'S2_B12', 'S2_NDVI']).select(['S2_B2', 'S2_B3', 'S2_B4', 'S2_B8', 'S2_B11', 'S2_B12', 'S2_NDVI'], ['S2_ref_B2', 'S2_ref_B3', 'S2_ref_B4', 'S2_ref_B8', 'S2_ref_B11', 'S2_ref_B12', 'S2_ref_NDVI']),
        cropland_mask,
        cropland_density,
        time_gap_img,
        s2_target_img.select(['S2_B2', 'S2_B3', 'S2_B4', 'S2_B8', 'S2_B11', 'S2_B12', 'S2_NDVI', 'S2_cloud_prob']).select(['S2_B2', 'S2_B3', 'S2_B4', 'S2_B8', 'S2_B11', 'S2_B12', 'S2_NDVI', 'S2_cloud_prob'], ['S2_target_B2', 'S2_target_B3', 'S2_target_B4', 'S2_target_B8', 'S2_target_B11', 'S2_target_B12', 'S2_target_NDVI', 'S2_target_cloud_prob'])
    ])
    
    return stack.updateMask(cropland_mask.gte(1))

print('Literature-aligned multiband stacker ready.')

In [ ]:
# Step 8: Pre-filtered Patch Export Task to Google Drive 
sample_s2 = s2_col.first()
sample_stack = build_triplet(sample_s2)

if sample_stack is not None:
    task = ee.batch.Export.image.toDrive(
        image=sample_stack,
        description='Faisalabad_Crop_Multiband_Corrected',
        folder=DRIVE_FOLDER,
        fileNamePrefix='faisalabad_multiband_corrected_2024',
        scale=SCALE,
        region=aoi,
        fileFormat='TFRecord',
        formatOptions={
            'patchDimensions': [PATCH_SIZE, PATCH_SIZE],
            'compressed': True
        }
    )
    task.start()
    print(f'Export task started to Google Drive folder "{DRIVE_FOLDER}". Status: {task.status()["state"]}')
else:
    print('Sample stack returned None (no clear match found).')